# SALSA training scheme
Based on the paper: https://openreview.net/forum?id=GEUVFvUYoc

The better the base model → the better the zero-shot performance → the better the SALSA-trained model’s performance.
This method has been proven to be one of the best ways to harness instruction-tuned LLMs. It performs well even with a small number of samples.

In this competition, the approach has an advantage because it works well out of the box without requiring extensive hyperparameter tuning. This is especially important here, since the few-shot examples are a valuable source for model tuning but are not directly accessible for hyperparameter optimization.

#Lets start with constant and prompt mapping definition

In [1]:
%%writefile jigsaw_const.py
epochs = 1
lr = 3E-4
max_seq_length = 1028 
load_in_4bit = False  
save_steps = 50
save_total_limit = 2  # multiple adpters can be used for stable estimator
        
SUB_PATH  = None
OUT_SUB_PATH  = "submission1.csv"
Q = 0.955
BASE_MODEL ="/kaggle/input/qwen-qwen3-4b-instruct-2507"
MODEL_NAME = "adapter"

seed = 52
TRAIN_PATH = '/kaggle/input/jigsaw-agile-community-rules/train.csv'
TEST_PATH = '/kaggle/input/jigsaw-agile-community-rules/test.csv'

EOT_MARKER = "<|im_end|>"

ANSWER_TEMPLATE = "<ANSWER>"

TASK_TEMPLATE = \
"""
/no_think
Given the comment from reddit:
<COMMENT>
{body}
</COMMENT>

Given rules:
<RULES>
{rule}
</RULES>

Does the comment violate any rules?

provide answer in format:
<ANSWER>#Number</ANSWER>

where the number is one of the following:
0 - No
1 - Yes
"""

classes = ['0', '1']

Writing jigsaw_const.py


In [2]:
%%writefile jigsaw_common.py
# -*- coding: utf-8 -*-

import pandas as pd
from datasets import load_dataset, Dataset
from datasets import load_dataset
import torch
import pandas as pd
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
import numpy as np
import os
from tqdm import tqdm
import pandas as pd
from jigsaw_const import *

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

TARGET_TOKENS = classes
TARGET_TOKENS_IDS = [tokenizer.encode(c)[0] for c in classes]
print(f"Class tokens {TARGET_TOKENS}")
print(f"Class token ids {TARGET_TOKENS_IDS}")

def train_weights_by_rule_default(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    col: str = "rule",
    default_missing: float = 0.1,
    include_nan: bool = True,
) -> pd.Series:
    # counts
    train_counts = train_df[col].value_counts(dropna=include_nan)
    test_counts  = test_df[col].value_counts(dropna=include_nan)

    # align test counts to train classes (missing -> 0)
    aligned_test = test_counts.reindex(train_counts.index).fillna(0)

    # class weights = test_count / train_count
    class_w = aligned_test / train_counts * len(train_df) / len(test_df)

    # train-only classes -> default
    class_w = class_w.mask(aligned_test == 0, default_missing)

    # map to rows (nonexistent keys -> default)
    weights = train_df[col].map(class_w).fillna(default_missing)

    # special case: NaN category (map won't match NaN)
    if include_nan and train_df[col].isna().any() and train_counts.index.isna().any():
        test_nan = test_counts.get(float("nan"), 0)
        train_nan = train_counts.get(float("nan"))
        nan_w = (test_nan / train_nan) if (train_nan and test_nan) else (
            default_missing if train_nan else default_missing
        )
        weights.loc[train_df[col].isna()] = nan_w

    return weights
    

def melt_examples(df):
        # Assuming your DataFrame is named df
    example_columns = ['positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']

    df_no_body = df.drop(columns=['body'])

    # Melt the DataFrame to long format with one row per example
    df_melted = df_no_body.melt(
        id_vars=['subreddit', 'rule'],
        value_vars=example_columns,
        var_name='example_type',
        value_name='body'
    )

    # Assign rule_violation based on example_type
    df_melted['rule_violation'] = df_melted['example_type'].apply(lambda x: 1 if 'positive' in x else 0)

    # Drop the example_type column if no longer needed
    df_final = df_melted.drop(columns=[
        'example_type'])  # .drop_duplicates() #Maybe instead just drop duplicate a majority vote for same 'subreddit', 'rule', 'body'
    return df_final
SANITY_TEST = len(pd.read_csv(TEST_PATH))<100 
def load_dataframe(path_test, path_train):

    df_train = pd.read_csv(path_train)
    df = pd.read_csv(path_test)
    original_body = df_train[['subreddit', 'rule', 'body', 'rule_violation']]

    df_final = pd.concat([melt_examples(df),melt_examples(df_train), original_body], ignore_index=True)
    df_final = df_final.drop(columns=['subreddit'])
    # # Majority voting
    df_final = df_final.reset_index(drop=True)
    df_final = (
        df_final.groupby([ 'rule', 'body'], as_index=False)
        .agg(
            rule_violation=('rule_violation',
                            lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
            score=('rule_violation',
                    lambda s: pd.to_numeric(s, errors='coerce').mean())
        )
    )
    

    if SUB_PATH is not None and os.path.exists(SUB_PATH):
        print("found submission source")
        sub_df = pd.read_csv(SUB_PATH)
        # Left-merge keeps all test rows; adjust to "inner" if you want only intersecting row_ids
        merged = df.merge(sub_df, on="row_id", how="left")
        # Keep only the requested columns that are present
        wanted = ["body", "rule", "rule_violation"]
        final_cols = [c for c in wanted if c in merged.columns]
        df_eval = merged[final_cols]
        df_eval["score"] = df_eval["rule_violation"]
        
        df_eval = df_eval.sort_values(
            "rule_violation",
            key=lambda s: (pd.to_numeric(s, errors="coerce") - 0.5).abs(),
            ascending=True,
            na_position="last",
        )
        df_eval = df_eval.head(FIRST_STAGE_PSUDO_LABELS)
        print(f"new records {len(df_eval)}")

        dupes = df_final.sample(frac=upsample_ratio, random_state=43)

        df_final = pd.concat([df_final,dupes,df_eval], ignore_index=True)

    else:
        print("not found prior submission source")

    
    # Optionally, sort or reset index
    df_final = df_final.reset_index(drop=True)
    #Rebalance weight - Haven't used this at the end
    weights = train_weights_by_rule_default(df_final, df, col="rule", default_missing=0.1)
    df_final = df_final.assign(weight=weights)

    shuffled_df = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    if SANITY_TEST:
        train_df = shuffled_df.head(200)
        print(train_df)
    else:
        train_df = shuffled_df
    return train_df

train_df = load_dataframe(TEST_PATH, TRAIN_PATH)
SANITY_TEST = len(train_df)<500
if SANITY_TEST:
    save_steps = 2
    epochs = 1


def compile_dataframe(df):
    columns = [ 'rule', 'body', 'rule_violation','score','weight']

    dataset = [
        ([
            {"role": "user", "content": TASK_TEMPLATE.format(
                                                             rule=rule,
                                                             body=body)},
            {"role": "assistant", "content": ANSWER_TEMPLATE + classes[int(target)]}], score, weight)
        for  rule, body, target, score, weight in df[columns].values]
    return dataset

dataset = compile_dataframe(train_df)

def formatting(dataset):
    texts = []
    score = []
    weight = []
    for i in range(len(dataset)):
        texts.append(tokenizer.apply_chat_template(dataset[i][0], tokenize=False, add_generation_prompt=False))
        score.append(dataset[i][1])
        weight.append(dataset[i][2])
    return Dataset.from_dict({'text': texts,'score': score, 'weight': weight})


dataset = formatting(dataset)
eot_ids=[tokenizer.convert_tokens_to_ids(EOT_MARKER)]

def _find_last_subsequence(haystack_ids, needle_ids):
    """Return start index of last occurrence of `needle_ids` (list[int]) in `haystack_ids`, or -1."""
    n = len(needle_ids)
    if n == 0:
        return -1
    for i in range(len(haystack_ids) - n, -1, -1):
        if haystack_ids[i:i+n] == needle_ids:
            return i
    return -1

def _find_last_eot_index(input_ids_tensor, eot_ids):
    """
    Find the last position of any EOT marker.
    For single-ID markers, returns the index of that ID.
    For subsequences, returns the index of the subsequence start.
    """
    ids = input_ids_tensor.tolist()
    last = -1
    for eot in eot_ids:
        if isinstance(eot, (list, tuple)):
            idx = _find_last_subsequence(ids, list(eot))
        else:
            # single token
            try:
                idx = len(ids) - 1 - ids[::-1].index(eot)
            except ValueError:
                idx = -1
        if idx > last:
            last = idx
    return last

def custom_train_on_last_response_token(example, mode="learn"):
    # Tokenize full text
    tokenized = tokenizer(example["text"], return_tensors="pt", add_special_tokens=False)
    input_ids = tokenized["input_ids"][0]

    # Labels: -100 everywhere except last token
    labels = [-100] * len(input_ids)

    # Locate the last end-of-turn token (or subsequence start)
    last_eot_idx = _find_last_eot_index(input_ids, eot_ids)

    # We want to label the token immediately BEFORE the EOT
    label_pos = last_eot_idx - 1

    labels[label_pos] = input_ids[label_pos]  # Keep only last token in assistant response

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "mode": int(mode == "learn"),
        "score": example.get("score",0.0),
        "weight": example.get("weight",0.1)
    }
    
tokenized_dataset = dataset.map(custom_train_on_last_response_token)
if SANITY_TEST:
    print(tokenized_dataset)
    print("#############################")
    print(tokenized_dataset["score"])
    print("#############################")
    
    #print sample
    print(tokenized_dataset[10])

# === Load tokenizer and get token IDs ===
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
target_token_ids = [tokenizer.encode(t, add_special_tokens=False)[0] for t in TARGET_TOKENS]

# === Prompt formatting ===
def make_prompt(example):
    messages = [
        {"role": "user", "content": TASK_TEMPLATE.format(**example)},
        {"role": "assistant", "content": ANSWER_TEMPLATE}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False,add_generation_prompt=False)

Writing jigsaw_common.py


In [3]:
%%writefile jigsaw_training.py
"""
Training code
"""
import os; os.environ["UNSLOTH_DISABLE_AUTO_UPDATES"] = "1"
local_rank = int(os.environ.get("LOCAL_RANK", "0"))
if local_rank!=0:
    import time
    time.sleep(60)#bugs in unsloth patching while running in multiprocess, I added sleep in hope to avoid collision
    
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

import torch
import re
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LlamaForSequenceClassification
from trl import SFTConfig, SFTTrainer
import pandas as pd
import numpy as np
from jigsaw_common import *

from transformers import TrainingArguments, DataCollatorForSeq2Seq


import torch
import torch.nn.functional as F


import os
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from transformers import TrainerCallback, default_data_collator
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import TrainingArguments
import numpy as np
from sklearn.metrics import roc_auc_score

class SALSATrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels", None)
        modes = inputs.pop("mode", None)
        scores = inputs.pop("score", None)
        weights = inputs.pop("weight", None)
        outputs = model(**inputs)
        logits = outputs.logits  # [B, T, V]

        if labels is None:
            loss = outputs.get("loss", None) if isinstance(outputs, dict) else None
            return (loss, outputs) if return_outputs else loss

        # --- Select only the target classes in the logits ---
        # shift for causal language modeling
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        # ensure same device
        device = shift_logits.device
        shift_labels = shift_labels.to(device)

        # Build a remap table: vocab_id -> class_index (0..K-1), others -> -100 (ignore)
        target_ids = torch.as_tensor(TARGET_TOKENS_IDS, device=device, dtype=torch.long)
        K = target_ids.numel()

        # remap has length vocab_size; fill with -100
        remap = torch.full((shift_logits.size(-1),), -100, dtype=torch.long, device=device)
        remap[target_ids] = torch.arange(K, dtype=torch.long, device=device)

        # Avoid indexing with -100: clamp negatives to 0 for the take, then restore ignore
        pad_mask = shift_labels.eq(-100)
        shift_labels_safe = shift_labels.clamp_min(0)
        mapped_labels = remap[shift_labels_safe]
        mapped_labels[pad_mask] = -100  # keep padding ignored

        # Now slice logits down to the K classes
        shift_logits_K = shift_logits.index_select(dim=-1, index=target_ids)


        nonpad_mask = shift_labels.ne(-100) 
        keep_idx = nonpad_mask.nonzero(as_tuple=True)
        logits_kept = shift_logits_K[keep_idx]  # shape: (N_kept, K)
        log_probs = F.log_softmax(logits_kept, dim=-1)
        probs = log_probs.exp()
        t=Q-1
        loss = 1/t*(1-scores*probs[:,1]**t-(1-scores)*probs[:,0]**t).mean()

        return (loss, outputs) if return_outputs else loss



if __name__ == "__main__":
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    os.environ.setdefault("NCCL_IB_DISABLE", "1")
    os.environ.setdefault("NCCL_ASYNC_ERROR_HANDLING", "1")

    model_base, tokenizer = FastLanguageModel.from_pretrained(
        model_name = BASE_MODEL,
        max_seq_length = max_seq_length,
        load_in_4bit = load_in_4bit,
        low_cpu_mem_usage = True,
        device_map = {"": f"cuda:{local_rank}"},
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"


    model = FastLanguageModel.get_peft_model(
        model_base,
        r = 32, 
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
        lora_alpha = 16,
        lora_dropout = 0.05,    
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,  
        loftq_config = None, 
    )
    
    model.gradient_checkpointing_enable()

    base_collator = DataCollatorForSeq2Seq(tokenizer)

    def collate_with_extras(features):
        pruned = []
        for f in features:
            f = dict(f)  # shallow copy so we don't mutate the dataset cache
            f.pop("text", None)  # drop if present
            pruned.append(f)

        batch = base_collator(pruned)

        return batch
    
    trainer = SALSATrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset=tokenized_dataset,
        dataset_text_field = None,
        max_seq_length = max_seq_length,
        data_collator = collate_with_extras,
        dataset_num_proc = 4,
        packing = False, 
        args = TrainingArguments(
            remove_unused_columns=False,
            gradient_checkpointing=True,
            ddp_find_unused_parameters=False,
            # packing=True,
            per_device_train_batch_size = 4,
            gradient_accumulation_steps = 4, 
            warmup_steps = 5,
            num_train_epochs = epochs,
            learning_rate = lr,
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 10,
            optim = "paged_adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
            report_to = "none",
            save_strategy = "steps",         
            save_steps = save_steps,              
            save_total_limit = save_total_limit,            
        ),
    )

    trainer.train()

Writing jigsaw_training.py


In [4]:
%%writefile /kaggle/working/accelerate_config.yaml
compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
num_machines: 1
machine_rank: 0
num_processes: 2
gpu_ids: all
mixed_precision: fp16

Writing /kaggle/working/accelerate_config.yaml


The training have to be on seperate process to release resources gracefully, else there is a resource reduction (partialy occupied from previous step). this also allow to run unsloth with accelerate on 2 blades

# VLLM Inference script

In [5]:
%%writefile infer_with_vllm.py
import os
import re
from jigsaw_common import *
from vllm import EngineArgs, LLMEngine
from vllm.lora.request import LoRARequest
from vllm import SamplingParams

# === Dataset class ===
from torch.utils.data import Dataset, DataLoader

def build_lora_requests(output_dir: str, prefix="checkpoint-", base_name="adapter") -> list:
    """
    Build a list of LoRARequest objects from LoRA adapter directories.

    Args:
        output_dir (str): Base directory containing LoRA checkpoints.
        prefix (str): Prefix for LoRA checkpoint dirs (default: "lora-checkpoint-").
        base_name (str): Base name for lora_name field (default: "adapter").

    Returns:
        list of LoRARequest
    """
    # Match directories like "lora-checkpoint-100"
    dirs = [
        d for d in os.listdir(output_dir)
        if os.path.isdir(os.path.join(output_dir, d)) and d.startswith(prefix)
    ]

    # Sort by step number
    dirs = sorted(
        dirs,
        key=lambda d: int(re.search(rf"{prefix}(\d+)", d).group(1))
    )

    # Build LoRARequest list
    lora_requests = []
    for idx, d in enumerate(dirs, start=1):
        lora_requests.append(LoRARequest(
            lora_name=f"{base_name}{idx}",
            lora_int_id=idx,
            lora_path=os.path.join(output_dir, d)
        ))

    return lora_requests



class PromptDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = make_prompt(row)
        return {
            "id": row.get("row_id", idx),
            "prompt": prompt,
            "meta": row.to_dict(),
        }

# Merge multiple predictions into single one, combination of the adapters.
import math

def product_groups(data, n):
    result = []
    for i in range(0, len(data), n):
        chunk = data[i:i+n]
        if len(chunk) == n:
            pos = sum(chunk)
            neg = sum([1-v for v in chunk])
            result.append(pos/n)
    return result


if __name__ == '__main__':
    engine = LLMEngine.from_engine_args(EngineArgs(
        model= BASE_MODEL,   # or another AWQ HF model
        enable_lora=True,
        max_lora_rank=32,
        trust_remote_code=True,
        enforce_eager=True,
        max_model_len=max_seq_length,
        dtype="half", # kv_cache_dtype="fp16",
        tensor_parallel_size=2,  # required: disable tensor parallel
        enable_prefix_caching=True,  
        # pipeline_parallel_size=2,  # optional: use pipeline parallel
    ))
    
    target_token_ids = []
    for token_str in TARGET_TOKENS:
        token_ids = tokenizer.encode(token_str, add_special_tokens=False)
        if len(token_ids) != 1:
            raise ValueError(f'"{token_str}" is not a single token: {token_ids}')
        target_token_ids.append(token_ids[0])
    
    # === Sampling params ===
    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=1,
        # logprobs=0,  # avoids triggering top-N full vocab softmax
        # prompt_logprobs=False,
        logprobs=len(TARGET_TOKENS),
        # prompt_logprobs=2,
        allowed_token_ids=target_token_ids
    )
    
    sample = tokenized_dataset[0]
    
    # Print raw input_ids and decoded text
    print("=== Input IDs ===")
    print(sample["input_ids"])
    print("=== Decoded Input ===")
    print(tokenizer.decode(sample["input_ids"]))
    
    # Print label mask
    print("=== Labels ===")
    print(sample["labels"])
    
    # Optionally inspect the actual target token (non -100)
    for i, (inp_id, label) in enumerate(zip(sample["input_ids"], sample["labels"])):
        if label != -100:
            print(f"Target token position: {i}")
            print(f"Input token: {tokenizer.decode([inp_id])}")
            print(f"Label token: {tokenizer.decode([label])}")
    
    
    
    output_dir = "outputs"
    lora_requests = build_lora_requests(output_dir)
    print(f"loaded {len(lora_requests)} adapters")
    
    
    CSV_PATH_TEST = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
    # === Load dataset ===
    df = pd.read_csv(CSV_PATH_TEST)
    df_sorted = df.sort_values("rule")

    dataset = PromptDataset(df_sorted)
    
    #Sanity check
    sample = dataset[0]
    
    # Print raw input_ids and decoded text
    print("=== templated prompt sample===")
    print(sample["prompt"])
    
    loader = DataLoader(dataset, batch_size=2, shuffle=False)
    
    
    # === Inference loop ===
    all_results = []
    rule_violations = []
    rule_violations_hard = []
    
    for batch in loader:
        prompts =[x[:-len("<|im_end|>\n")] for x in batch["prompt"]]
        ids = batch["id"]
        metas = batch["meta"]
        for q, prompt in enumerate(prompts):
            for m, lora_request in enumerate(lora_requests):
                engine.add_request(f"{ids[q]}_{m}", prompt, sampling_params, lora_request=lora_request)
            
    # Assuming you know total number of prompts sent to engine
    num_total_requests = len(dataset)  # Set this to the total number of prompts
    progress_bar = tqdm(total=num_total_requests, desc="Processing Requests")
    
    rule_violations = []
    rule_violations_hard = []
    ids = []
    while engine.has_unfinished_requests():
        output = engine.step()
    
        for out in output:
            q = out.request_id
            result = out.outputs[0]
    
            text = result.text.strip()
            prob = np.exp(result.logprobs[0][target_token_ids[0]].logprob)  # token index 15 assumed
            lz = prob
            lo = 1 - prob
            rule_violations.append(lo)
            rule_violations_hard.append(lo > 0.5)
            ids.append(q)
            progress_bar.update(1)  # Update for each processed request
    
    progress_bar.close()

    
    
    ids_int = [int(x.split("_")[0]) for x in ids]
    # Sort both based on string_list
    paired = sorted(zip(ids_int, rule_violations))  # sort by string values
    
    # Split back into two separate lists
    ids_int, rule_violations = zip(*paired)
    
    n = len(lora_requests)
    rule_violations_mean = product_groups(rule_violations, n)  
    map_results = {k:v for k,v in zip(ids_int[::n],rule_violations_mean)}
    
    df["rule_violation"] = df["row_id"].map(map_results)
    
    df.to_csv(OUT_SUB_PATH, index=False, columns=["row_id", "rule_violation"])

Writing infer_with_vllm.py


# Logic flow

In [6]:
%%bash
export HF_HUB_OFFLINE=1
export TRANSFORMERS_OFFLINE=1
export HF_DATASETS_OFFLINE=1
export HF_HUB_DISABLE_TELEMETRY=1
export UNSLOTH_DISABLE_AUTO_UPDATES=1
export ACCELERATE_CONFIG_FILE=/kaggle/working/accelerate_config.yaml
export NCCL_IB_DISABLE=1
export NCCL_ASYNC_ERROR_HANDLING=1

accelerate launch jigsaw_training.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-30 18:29:09 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 10-30 18:29:09 [__init__.py:239] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
Class tokens ['0', '1']
Class token ids [15, 16]
not found prior submission source
                                                  rule  ...    weight
0    No legal advice: Do not offer or request legal...  ...  0.185094
1    No legal advice: Do not offer or request legal...  ...  0.185094
2    No Advertising: Spam, referral links, unsolici...  ...  1.957657
3    No Advertising: Spam, referral links, unsolici...  ...  1.957657
4    No legal advice: Do not offer or request legal...  ...  0.185094
..                                                 ...  ...       ...
195  No legal advice: Do not offer or request legal...  ...  0.185094
196  No Advertising: Spam, referral links, unsolici..

[W1030 18:28:25.823136841 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1030 18:28:33.825436670 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2025-10-30 18:28:46.104607: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761848926.476397      62 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761848926.586056      62 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]2025-10-30 18:29:40.419795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register fac

In [7]:
!python infer_with_vllm.py

Class tokens ['0', '1']
Class token ids [15, 16]
not found prior submission source
                                                  rule  ...    weight
0    No legal advice: Do not offer or request legal...  ...  0.185094
1    No legal advice: Do not offer or request legal...  ...  0.185094
2    No Advertising: Spam, referral links, unsolici...  ...  1.957657
3    No Advertising: Spam, referral links, unsolici...  ...  1.957657
4    No legal advice: Do not offer or request legal...  ...  0.185094
..                                                 ...  ...       ...
195  No legal advice: Do not offer or request legal...  ...  0.185094
196  No Advertising: Spam, referral links, unsolici...  ...  1.957657
197  No Advertising: Spam, referral links, unsolici...  ...  1.957657
198  No legal advice: Do not offer or request legal...  ...  0.185094
199  No legal advice: Do not offer or request legal...  ...  0.185094

[200 rows x 5 columns]
Map: 100%|████████████████████████████| 200/200 [00:0

# Train with Psudo Labels
we shall use here the larger model as the compute budget allow to run only one step with it

In [8]:
%%writefile jigsaw_const.py
epochs = 1
lr = 3E-4
max_seq_length = 1024 
load_in_4bit = False  
save_steps = 50
save_total_limit = 1
Q = 0.955 
SUB_PATH  = "submission1.csv"
OUT_SUB_PATH = "submission2.csv"
FIRST_STAGE_PSUDO_LABELS = 8000
upsample_ratio=0.6

BASE_MODEL = "/kaggle/input/qwen-3/transformers/14b-awq/1"
MODEL_NAME = "adapter"

seed = 52
TRAIN_PATH = '/kaggle/input/jigsaw-agile-community-rules/train.csv'
TEST_PATH = '/kaggle/input/jigsaw-agile-community-rules/test.csv'

EOT_MARKER = "<|im_end|>"

ANSWER_TEMPLATE = "<ANSWER>"

TASK_TEMPLATE = \
"""
Does the comment violate the rule? Answer in the format:
<ANSWER>#Number</ANSWER>
Where the number is one of:
0 - No
1 - Yes

The rule:
<RULE>
{rule}
</RULE>

The Comment:
<COMMENT>
{body}
</COMMENT>
"""

classes = ['0', '1']

Overwriting jigsaw_const.py


In [9]:
%%bash
export HF_HUB_OFFLINE=1
export TRANSFORMERS_OFFLINE=1
export HF_DATASETS_OFFLINE=1
export HF_HUB_DISABLE_TELEMETRY=1
export UNSLOTH_DISABLE_AUTO_UPDATES=1
export ACCELERATE_CONFIG_FILE=/kaggle/working/accelerate_config.yaml
export NCCL_IB_DISABLE=1
export NCCL_ASYNC_ERROR_HANDLING=1

accelerate launch jigsaw_training.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-30 18:36:44 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 10-30 18:36:44 [__init__.py:239] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
Class tokens ['0', '1']
Class token ids [15, 16]
found submission source
new records 10
                                                  rule  ...    weight
0    No Advertising: Spam, referral links, unsolici...  ...  1.974490
1    No Advertising: Spam, referral links, unsolici...  ...  1.974490
2    No Advertising: Spam, referral links, unsolici...  ...  1.974490
3    No Advertising: Spam, referral links, unsolici...  ...  1.974490
4    No legal advice: Do not offer or request legal...  ...  0.183761
..                                                 ...  ...       ...
195  No Advertising: Spam, referral links, unsolici...  ...  1.974490
196  No legal advice: Do not offer or request le

[W1030 18:36:20.285868301 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1030 18:36:28.288287431 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2025-10-30 18:36:36.953919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761849396.975690     598 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761849396.982325     598 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/kaggle/working/jigsaw_common.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docum

In [10]:

!python infer_with_vllm.py

Class tokens ['0', '1']
Class token ids [15, 16]
found submission source
/kaggle/working/jigsaw_common.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_eval["score"] = df_eval["rule_violation"]
new records 10
                                                  rule  ...    weight
0    No Advertising: Spam, referral links, unsolici...  ...  1.974490
1    No Advertising: Spam, referral links, unsolici...  ...  1.974490
2    No Advertising: Spam, referral links, unsolici...  ...  1.974490
3    No Advertising: Spam, referral links, unsolici...  ...  1.974490
4    No legal advice: Do not offer or request legal...  ...  0.183761
..                                                 ...  ...       ...
195  No Advertising: Spam, referral links, unsoli

# Merge 2 stages for enhanced accuracy (instead of taking only the last)

In [11]:
import pandas as pd

def load_and_collapse(path):
    df = pd.read_csv(path, usecols=["row_id", "rule_violation"])
    # Ensure correct types and collapse duplicates by mean
    df["rule_violation"] = pd.to_numeric(df["rule_violation"], errors="coerce")
    df = df.dropna(subset=["row_id"]).groupby("row_id", as_index=False)["rule_violation"].mean()
    return df

def merge_results(path1, path2, out_path="submission.csv", w1=0.7, w2=0.3):
    df1 = load_and_collapse(path1).rename(columns={"rule_violation": "rv1"})
    df2 = load_and_collapse(path2).rename(columns={"rule_violation": "rv2"})

    merged = pd.merge(df1, df2, on="row_id", how="outer")

    # Dynamic weights to handle missing values
    w1_eff = merged["rv1"].notna().astype(float) * w1
    w2_eff = merged["rv2"].notna().astype(float) * w2
    denom = w1_eff + w2_eff

    # Weighted average; if both missing -> NaN (rare; row would come from nowhere)
    merged["rule_violation"] = (merged["rv1"].fillna(0) * w1_eff + merged["rv2"].fillna(0) * w2_eff) / denom.replace(0, pd.NA)

    # If a row_id exists only in one file, rule_violation becomes that file's value
    merged.loc[denom == 0, "rule_violation"] = pd.NA

    out = merged[["row_id", "rule_violation"]].copy()


    out.to_csv(out_path, index=False)
    print(f"Wrote {len(out)} rows to {out_path}")

In [12]:
merge_results("submission2.csv","submission1.csv")

Wrote 10 rows to submission.csv
